# Assignment 1: Introduction to Genetic Algorithms

## 🎯 Learning Objectives

In this assignment, you will:
- Understand the basic components of a Genetic Algorithm
- Implement population initialization
- Code fitness evaluation
- Create selection, crossover, and mutation operators
- Build a complete GA from scratch
- Optimize benchmark functions

This assignment is designed to give you hands-on experience with GAs. Let's get started!

---

## 📦 1. Import Libraries

Run the cell below to import the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import visualization toolkit
sys.path.append('../ga_toolkit')
from visualization import plot_convergence

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

**Expected Output:**
```
✅ Libraries imported successfully!
NumPy version: 1.x.x
```

---

## 🧬 2. Population Initialization

The first step in a GA is to create an initial population. Each individual (chromosome) represents a potential solution.

### 2.1 - Binary Encoding

**Exercise 1:** Implement the `initialize_population_binary` function.

**Instructions:**
- Create a 2D array of shape `(pop_size, n_variables)`
- Each element should be randomly 0 or 1
- Use `np.random.randint()`

**Hints:**
- `np.random.randint(low, high, size)` generates random integers

In [ ]:
def initialize_population_binary(pop_size, n_variables):
    """
    Initialize a binary population.
    
    Arguments:
    pop_size -- number of individuals in population
    n_variables -- number of variables (genes) per individual
    
    Returns:
    population -- numpy array of shape (pop_size, n_variables) with binary values
    """
    
    ### START CODE HERE ### (≈ 1 line)
    population = np.random.randint(0, 2, size=(pop_size, n_variables))
    ### END CODE HERE ###
    
    return population

In [ ]:
# Test your implementation
pop = initialize_population_binary(5, 8)

print("Population:")
print(pop)
print(f"\nShape: {pop.shape}")
print(f"All values are 0 or 1: {np.all((pop == 0) | (pop == 1))}")

# Assertions
assert pop.shape == (5, 8), f"Expected shape (5, 8), got {pop.shape}"
assert np.all((pop == 0) | (pop == 1)), "Population should only contain 0s and 1s"
assert pop.dtype == np.int64 or pop.dtype == np.int32, "Population should be integer type"

print("\n✅ All tests passed!")

**Expected Output:**
```
Population:
[[0 1 1 0 1 0 0 1]
 [1 0 1 1 0 1 0 0]
 ...]

Shape: (5, 8)
All values are 0 or 1: True

✅ All tests passed!
```

### 2.2 - Real-Valued Encoding

**Exercise 2:** Implement the `initialize_population_real` function.

**Instructions:**
- Create a population with real values in the range defined by `bounds`
- `bounds` is a list of tuples: `[(min1, max1), (min2, max2), ...]`
- Use `np.random.uniform()` for each variable

In [ ]:
def initialize_population_real(pop_size, bounds):
    """
    Initialize a real-valued population.
    
    Arguments:
    pop_size -- number of individuals
    bounds -- list of (min, max) tuples for each variable
    
    Returns:
    population -- numpy array of shape (pop_size, n_variables)
    """
    
    n_variables = len(bounds)
    population = np.zeros((pop_size, n_variables))
    
    ### START CODE HERE ### (≈ 3-4 lines)
    for i in range(n_variables):
        min_val, max_val = bounds[i]
        population[:, i] = np.random.uniform(min_val, max_val, pop_size)
    ### END CODE HERE ###
    
    return population

In [ ]:
# Test your implementation
bounds = [(-5, 5), (-10, 10), (0, 1)]
pop = initialize_population_real(5, bounds)

print("Population:")
print(pop)
print(f"\nShape: {pop.shape}")

# Check bounds
print(f"\nVariable 0 in range [-5, 5]: {np.all((pop[:, 0] >= -5) & (pop[:, 0] <= 5))}")
print(f"Variable 1 in range [-10, 10]: {np.all((pop[:, 1] >= -10) & (pop[:, 1] <= 10))}")
print(f"Variable 2 in range [0, 1]: {np.all((pop[:, 2] >= 0) & (pop[:, 2] <= 1))}")

# Assertions
assert pop.shape == (5, 3), f"Expected shape (5, 3), got {pop.shape}"
assert np.all((pop[:, 0] >= -5) & (pop[:, 0] <= 5)), "Variable 0 out of bounds"
assert np.all((pop[:, 1] >= -10) & (pop[:, 1] <= 10)), "Variable 1 out of bounds"
assert np.all((pop[:, 2] >= 0) & (pop[:, 2] <= 1)), "Variable 2 out of bounds"

print("\n✅ All tests passed!")

**Expected Output:**
```
Population:
[[ 2.45 -3.21  0.87]
 [-1.23  8.45  0.12]
 ...]

Shape: (5, 3)

Variable 0 in range [-5, 5]: True
Variable 1 in range [-10, 10]: True
Variable 2 in range [0, 1]: True

✅ All tests passed!
```

---

## 🎯 3. Fitness Evaluation

The fitness function measures how good each solution is. We'll use standard benchmark functions.

### 3.1 - Sphere Function

The sphere function is:
$$f(x) = \sum_{i=1}^{n} x_i^2$$

**Optimum:** $f(0, 0, ..., 0) = 0$

**Exercise 3:** Implement the sphere function.

In [ ]:
def sphere_function(x):
    """
    Sphere function: sum of squares.
    
    Arguments:
    x -- numpy array (1D or 2D)
    
    Returns:
    f -- function value (scalar or array)
    """
    
    ### START CODE HERE ### (≈ 1 line)
    f = np.sum(x**2, axis=-1)
    ### END CODE HERE ###
    
    return f

In [ ]:
# Test your implementation
test_cases = [
    (np.array([0, 0, 0]), 0),
    (np.array([1, 1, 1]), 3),
    (np.array([2, -2, 2]), 12),
]

print("Testing sphere function:")
for x, expected in test_cases:
    result = sphere_function(x)
    print(f"  f({x}) = {result:.2f} (expected: {expected})")
    assert np.isclose(result, expected), f"Expected {expected}, got {result}"

print("\n✅ All tests passed!")

### 3.2 - Evaluate Population

**Exercise 4:** Evaluate fitness for an entire population.

**Instructions:**
- Apply the fitness function to each individual
- Return an array of fitness values
- For minimization problems, negate the fitness (GA maximizes)

In [ ]:
def evaluate_population(population, fitness_func, maximize=False):
    """
    Evaluate fitness for each individual in population.
    
    Arguments:
    population -- numpy array (pop_size, n_variables)
    fitness_func -- function to evaluate
    maximize -- if False, negate fitness (for minimization)
    
    Returns:
    fitness -- numpy array (pop_size,) with fitness values
    """
    
    ### START CODE HERE ### (≈ 3-4 lines)
    fitness = np.array([fitness_func(ind) for ind in population])
    if not maximize:
        fitness = -fitness
    ### END CODE HERE ###
    
    return fitness

In [ ]:
# Test your implementation
test_pop = np.array([
    [0, 0, 0],
    [1, 1, 1],
    [2, -2, 2]
])

fitness = evaluate_population(test_pop, sphere_function, maximize=False)

print("Population:")
print(test_pop)
print(f"\nFitness (negated for minimization):")
print(fitness)

expected = np.array([0, -3, -12])
assert np.allclose(fitness, expected), f"Expected {expected}, got {fitness}"

print("\n✅ Test passed!")

---

## 🎲 4. Selection

Selection chooses individuals to become parents based on fitness.

### 4.1 - Tournament Selection

**Exercise 5:** Implement tournament selection.

**Algorithm:**
1. Randomly select `tournament_size` individuals
2. Choose the one with best fitness
3. Repeat to select multiple parents

In [ ]:
def tournament_selection(population, fitness, n_parents, tournament_size=3):
    """
    Select parents using tournament selection.
    
    Arguments:
    population -- numpy array (pop_size, n_variables)
    fitness -- numpy array (pop_size,)
    n_parents -- number of parents to select
    tournament_size -- size of each tournament
    
    Returns:
    parents -- numpy array (n_parents, n_variables)
    """
    
    pop_size = population.shape[0]
    parents = []
    
    ### START CODE HERE ### (≈ 5-7 lines)
    for _ in range(n_parents):
        # Random tournament
        tournament_idx = np.random.choice(pop_size, tournament_size, replace=False)
        # Get fitness of tournament participants
        tournament_fitness = fitness[tournament_idx]
        # Winner is the one with best (highest) fitness
        winner_idx = tournament_idx[np.argmax(tournament_fitness)]
        parents.append(population[winner_idx])
    ### END CODE HERE ###
    
    return np.array(parents)

In [ ]:
# Test your implementation
np.random.seed(42)

test_pop = np.array([
    [1, 1],  # fitness = -2
    [0, 0],  # fitness = 0 (best)
    [2, 2],  # fitness = -8
    [1, 0],  # fitness = -1
])

test_fitness = np.array([-2, 0, -8, -1])

parents = tournament_selection(test_pop, test_fitness, n_parents=3, tournament_size=2)

print("Selected parents:")
print(parents)
print(f"\nShape: {parents.shape}")

assert parents.shape == (3, 2), f"Expected shape (3, 2), got {parents.shape}"
print("\n✅ Test passed!")

---

## 🧬 5. Crossover (Recombination)

Crossover combines genetic information from two parents.

### 5.1 - Single-Point Crossover

**Exercise 6:** Implement single-point crossover.

**Algorithm:**
1. Choose a random crossover point
2. Child 1 = Parent 1[0:point] + Parent 2[point:]
3. Child 2 = Parent 2[0:point] + Parent 1[point:]

In [ ]:
def single_point_crossover(parent1, parent2):
    """
    Perform single-point crossover.
    
    Arguments:
    parent1, parent2 -- numpy arrays (n_variables,)
    
    Returns:
    child1, child2 -- numpy arrays (n_variables,)
    """
    
    n_variables = len(parent1)
    
    ### START CODE HERE ### (≈ 4-5 lines)
    # Random crossover point (not at boundaries)
    point = np.random.randint(1, n_variables)
    
    # Create children
    child1 = np.concatenate([parent1[:point], parent2[point:]])
    child2 = np.concatenate([parent2[:point], parent1[point:]])
    ### END CODE HERE ###
    
    return child1, child2

In [ ]:
# Test your implementation
np.random.seed(42)

parent1 = np.array([1, 1, 1, 1, 1])
parent2 = np.array([0, 0, 0, 0, 0])

child1, child2 = single_point_crossover(parent1, parent2)

print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")
print(f"Child 1:  {child1}")
print(f"Child 2:  {child2}")

# Children should be different from parents (with high probability)
assert len(child1) == len(parent1), "Child length should match parent length"
assert len(child2) == len(parent2), "Child length should match parent length"

print("\n✅ Test passed!")

---

## 🎲 6. Mutation

Mutation introduces random changes to maintain diversity.

### 6.1 - Bit-Flip Mutation (Binary)

**Exercise 7:** Implement bit-flip mutation for binary chromosomes.

In [ ]:
def bit_flip_mutation(individual, mutation_rate=0.1):
    """
    Perform bit-flip mutation on binary individual.
    
    Arguments:
    individual -- numpy array (n_variables,) with binary values
    mutation_rate -- probability of flipping each bit
    
    Returns:
    mutated -- numpy array (n_variables,)
    """
    
    mutated = individual.copy()
    
    ### START CODE HERE ### (≈ 3-4 lines)
    for i in range(len(mutated)):
        if np.random.rand() < mutation_rate:
            mutated[i] = 1 - mutated[i]  # Flip bit
    ### END CODE HERE ###
    
    return mutated

In [ ]:
# Test your implementation
np.random.seed(42)

original = np.array([1, 1, 1, 1, 1, 1, 1, 1])
mutated = bit_flip_mutation(original, mutation_rate=0.3)

print(f"Original: {original}")
print(f"Mutated:  {mutated}")
print(f"\nNumber of flipped bits: {np.sum(original != mutated)}")

assert len(mutated) == len(original), "Length should be preserved"
assert np.all((mutated == 0) | (mutated == 1)), "Values should be binary"

print("\n✅ Test passed!")

---

## 🔄 7. Complete GA Loop

Now let's put it all together!

**Exercise 8:** Complete the main GA loop.

In [ ]:
def simple_genetic_algorithm(fitness_func, bounds, pop_size=50, max_generations=100,
                            mutation_rate=0.1, crossover_rate=0.8):
    """
    Simple Genetic Algorithm.
    
    Arguments:
    fitness_func -- function to optimize
    bounds -- list of (min, max) for each variable
    pop_size -- population size
    max_generations -- number of generations
    mutation_rate -- mutation probability
    crossover_rate -- crossover probability
    
    Returns:
    best_solution, best_fitness, history
    """
    
    n_variables = len(bounds)
    
    # Initialize
    population = initialize_population_real(pop_size, bounds)
    
    # History tracking
    history = {'best_fitness': [], 'mean_fitness': []}
    
    best_solution = None
    best_fitness = -np.inf
    
    for generation in range(max_generations):
        # Evaluate
        fitness = evaluate_population(population, fitness_func, maximize=False)
        
        # Track best
        gen_best_idx = np.argmax(fitness)
        if fitness[gen_best_idx] > best_fitness:
            best_fitness = fitness[gen_best_idx]
            best_solution = population[gen_best_idx].copy()
        
        history['best_fitness'].append(best_fitness)
        history['mean_fitness'].append(np.mean(fitness))
        
        ### START CODE HERE ### (≈ 15-20 lines)
        # Selection
        parents = tournament_selection(population, fitness, pop_size)
        
        # Create offspring
        offspring = []
        for i in range(0, pop_size, 2):
            parent1 = parents[i]
            parent2 = parents[min(i+1, pop_size-1)]
            
            # Crossover
            if np.random.rand() < crossover_rate:
                child1, child2 = single_point_crossover(parent1, parent2)
            else:
                child1, child2 = parent1.copy(), parent2.copy()
            
            # Mutation (for real-valued: Gaussian noise)
            if np.random.rand() < mutation_rate:
                child1 = child1 + np.random.normal(0, 0.1, n_variables)
            if np.random.rand() < mutation_rate:
                child2 = child2 + np.random.normal(0, 0.1, n_variables)
            
            offspring.append(child1)
            offspring.append(child2)
        
        # New population
        population = np.array(offspring[:pop_size])
        
        # Enforce bounds
        for i in range(n_variables):
            population[:, i] = np.clip(population[:, i], bounds[i][0], bounds[i][1])
        ### END CODE HERE ###
    
    return best_solution, best_fitness, history

---

## 🚀 8. Test Your GA!

Let's optimize the sphere function and visualize the results.

In [ ]:
# Run GA
bounds = [(-5, 5)] * 5  # 5 dimensions

best_sol, best_fit, history = simple_genetic_algorithm(
    fitness_func=sphere_function,
    bounds=bounds,
    pop_size=50,
    max_generations=100,
    mutation_rate=0.1,
    crossover_rate=0.8
)

print("Optimization Results:")
print("="*50)
print(f"Best solution: {best_sol}")
print(f"Best fitness: {best_fit:.6f}")
print(f"Actual function value: {-best_fit:.6f}")
print(f"Distance from optimum: {np.linalg.norm(best_sol):.6f}")
print("\n✅ GA completed successfully!")

In [ ]:
# Visualize convergence
plot_convergence(history, title="Sphere Function Optimization with GA")
plt.show()

**Expected Results:**
- Best solution should be close to [0, 0, 0, 0, 0]
- Best fitness should be close to 0 (or very small negative number)
- Convergence plot should show smooth improvement

**Congratulations! You've implemented a complete Genetic Algorithm from scratch!** 🎉

---

## 📊 9. What You've Learned

In this assignment, you:

✅ Implemented population initialization (binary and real-valued)  
✅ Created fitness evaluation functions  
✅ Coded tournament selection  
✅ Implemented single-point crossover  
✅ Created mutation operators  
✅ Built a complete GA from scratch  
✅ Optimized benchmark functions  
✅ Visualized GA convergence  

**Next Steps:**
- Try different benchmark functions (Rastrigin, Rosenbrock)
- Experiment with parameters (population size, mutation rate)
- Move on to Assignment 2 for advanced operators!

---

## 🎯 Optional Challenge

Try to:
1. Implement elitism (keep best solutions)
2. Add uniform crossover
3. Try the Rastrigin function (harder!)
4. Compare different selection methods

Good luck! 🚀